# Case Assignment #1
## Sarthak Jain, MSIS 510 B

In [28]:
# Add imports
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf


In [29]:
# Load the datasets
df_train = pd.read_csv("train_data-1.csv")
df_test  = pd.read_csv("test_data-1.csv")
df_site_const = pd.read_csv("site_const_data-1-1.csv")

In [30]:
# Rename columns to replace '.' with '_' to use in smf methods
for df in (df_train, df_test, df_site_const):   # use your own dataframe names
    df.columns = df.columns.str.replace('.', '_', regex=False)

#### Q1

In [31]:
testing_size = df_test.shape[0]
training_size = df_train.shape[0] 
print("Testing Set Size:", testing_size)
print("Training Set Size:", training_size)

total_sample_data = testing_size + training_size
print("Total Sample Size:", total_sample_data)

training_percentage = 100 * (training_size/total_sample_data)
testing_percentage =  100 * (testing_size/total_sample_data)

print("Training %:", training_percentage)
print("Testing %:", testing_percentage)


Testing Set Size: 85
Training Set Size: 374
Total Sample Size: 459
Training %: 81.48148148148148
Testing %: 18.51851851851852


#### Q2

In [32]:
# Linear Regression
linear_regression = smf.ols('annual_profit ~ agg_inc + sqft + col_grad + com60', data=df_train)
linear_regression_results = linear_regression.fit()

# Print model summary
print(linear_regression_results.summary())

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.786
Model:                            OLS   Adj. R-squared:                  0.784
Method:                 Least Squares   F-statistic:                     339.1
Date:                Tue, 03 Nov 2026   Prob (F-statistic):          3.73e-122
Time:                        15:34:53   Log-Likelihood:                -5107.5
No. Observations:                 374   AIC:                         1.022e+04
Df Residuals:                     369   BIC:                         1.024e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    8.36e+04   4.11e+04      2.035      0.0

In [33]:
print(linear_regression_results.params.apply(lambda x: f'{x:,.6f}'))

Intercept     83,597.092456
agg_inc            0.002807
sqft             383.363054
col_grad     346,821.138695
com60        218,265.764624
dtype: str


#### Q3

In [34]:
milagro_store_scenario = {
    'agg_inc': 100000000,
    'sqft': 800,
    'col_grad': 0.3,
    'com60': 0.1
}
annual_profitability = linear_regression_results.predict(milagro_store_scenario)
# print(annual_profitability)
print(f"Predicted profitability is: ${annual_profitability[0]:,.2f}")

Predicted profitability is: $796,894.99


#### Q4

In [35]:
# The R^2 for the training data. Need SSE and SST in order to calculate R^2
# a) R-squared value on training data
train_predictions = linear_regression_results.predict(df_train)
train_actual = df_train['annual_profit']

# Calculate SSE and SST for training data
train_mean = train_actual.mean()
train_sst = ((train_actual - train_mean) ** 2).sum()
train_sse = ((train_actual - train_predictions) ** 2).sum()

# Calculate R-squared for training data
train_r2 = 1 - (train_sse / train_sst)
print("Training data:")
print(f"R-squared: {train_r2:.4f}")


# The R^2 for the testing data. Need SSE and SST in order to calculate R^2
test_predictions = linear_regression_results.predict(df_test)
test_actual = df_test['annual_profit']

# Calculate SST and SSE for test data
test_sst = ((test_actual - train_mean) ** 2).sum()
test_sse = ((test_actual - test_predictions) ** 2).sum()

# Calculate R-squared for test data
test_r2 = 1 - (test_sse / test_sst)
print("Test data:")
print(f"R-squared: {test_r2:.4f}")

Training data:
R-squared: 0.7861
Test data:
R-squared: 0.7201


#### Q5

In [36]:
p_values = linear_regression_results.pvalues
print(p_values)

for feature, p_value in p_values.items():
    if p_value < 0.05:
        print(f"The feature '{feature}' is statistically significant (p-value: {p_value:.4f})")
    else:
        print(f"The feature '{feature}' is NOT statistically significant (p-value: {p_value:.4f})")

#Smallest p-value
smallest_p_value_feature = p_values.idxmin()
print(f"The feature with the smallest p-value is '{smallest_p_value_feature}' (p-value: {p_values[smallest_p_value_feature]:.4f})")
#Largest p-value
largest_p_value_feature = p_values.idxmax()
print(f"The feature with the largest p-value is '{largest_p_value_feature}' (p-value: {p_values[largest_p_value_feature]:.4f})")

Intercept    4.258413e-02
agg_inc      5.249611e-62
sqft         1.063204e-09
col_grad     2.309665e-03
com60        2.686647e-02
dtype: float64
The feature 'Intercept' is statistically significant (p-value: 0.0426)
The feature 'agg_inc' is statistically significant (p-value: 0.0000)
The feature 'sqft' is statistically significant (p-value: 0.0000)
The feature 'col_grad' is statistically significant (p-value: 0.0023)
The feature 'com60' is statistically significant (p-value: 0.0269)
The feature with the smallest p-value is 'agg_inc' (p-value: 0.0000)
The feature with the largest p-value is 'Intercept' (p-value: 0.0426)


#### Q6

In [37]:
exclude = ['state', 'store_number', 'annual_profit']
include = [col for col in df_train.columns if col not in exclude]

print("Features included in the model:")
for feature in include:
    print(f"  - {feature}")

print("Numerical predictor variables included in the model:")
for feature in include:
    if df_train[feature].dtype in ['int64', 'float64']:
        print(f"  - {feature}")




Features included in the model:
  - agg_inc
  - sqft
  - col_grad
  - com60
  - lci
  - nearcomp
  - nearmil
  - freestand
  - gini
  - housemed
Numerical predictor variables included in the model:
  - agg_inc
  - sqft
  - col_grad
  - com60
  - lci
  - nearcomp
  - nearmil
  - freestand
  - gini
  - housemed


In [38]:
correlation_matrix = df_train[include].corr()
print("Correlation matrix:")
print(correlation_matrix.round(5))


Correlation matrix:
           agg_inc     sqft  col_grad    com60      lci  nearcomp  nearmil  \
agg_inc    1.00000  0.51183   0.67019 -0.23883 -0.31483  -0.14758  0.16018   
sqft       0.51183  1.00000   0.35303 -0.05535 -0.29966  -0.07415  0.12176   
col_grad   0.67019  0.35303   1.00000 -0.22387 -0.31355  -0.18182  0.06533   
com60     -0.23883 -0.05535  -0.22387  1.00000 -0.00408   0.02477  0.03771   
lci       -0.31483 -0.29966  -0.31355 -0.00408  1.00000   0.07143 -0.11476   
nearcomp  -0.14758 -0.07415  -0.18182  0.02477  0.07143   1.00000 -0.11045   
nearmil    0.16018  0.12176   0.06533  0.03771 -0.11476  -0.11045  1.00000   
freestand  0.17311  0.15047   0.16615 -0.03201 -0.21953   0.10422  0.21137   
gini       0.06861  0.11095   0.00135 -0.03507 -0.10490   0.11227 -0.01217   
housemed   0.48639  0.17686   0.54693 -0.10569 -0.15198  -0.17459  0.03186   

           freestand     gini  housemed  
agg_inc      0.17311  0.06861   0.48639  
sqft         0.15047  0.11095   0.176

In [39]:
#The three pairs of variables with the strongest correlations (highest absolute values)
strongest_correlations = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        var1 = correlation_matrix.columns[i]
        var2 = correlation_matrix.columns[j]
        corr_value = correlation_matrix.iloc[i, j]
        strongest_correlations.append((var1, var2, abs(corr_value)))

# Sort by absolute correlation value in descending order and take the top 3
strongest_correlations.sort(key=lambda x: x[2], reverse=True)
top_3 = strongest_correlations[:3]

print("Three pairs of variables with the strongest correlations:")
for var1, var2, corr_value in top_3:
    print(f"  - {var1} and {var2}: {corr_value:.5f}")

Three pairs of variables with the strongest correlations:
  - agg_inc and col_grad: 0.67019
  - col_grad and housemed: 0.54693
  - agg_inc and sqft: 0.51183
